# Phase 8 — the error analysis (exploratory)

The unblinding is done and its verdicts are retained under the pre-registration's §9.
**Nothing in this notebook can change them.** Every table here is exploratory and
labelled post hoc. The analyses (P8.0–P8.6) were declared in
`docs/04_experiment_register.md` and committed with `scripts/phase8.py` before this
notebook first ran.

| Section | What | Reads labels? |
|---|---|---|
| 3 | **P8.1** — the secondary outcomes the unblinding computed but did not print: accuracy at 80/90% coverage, F3, F4, F5, D4, E1–E3 | no: `results.json` only |
| 4 | **P8.0** reproduction gate, then **G1** (what disagreement catches that confidence misses) and **G3** (where M3 contradicts a correct grade) | yes, again, after the unblinding |
| 5 | **P8.5** — figure-1 candidates, drawn with M2's lesion outlines | — |
| 6 | **P8.6** — Messidor-2's preprocessing against the other sources | no |

Section 4 refuses to print anything unless it reproduces every published AUC exactly.

## Inputs

`verify-dr-results` (10's output) · `verify-dr-locked` (09's) · `verify-dr-fitted` (08's) ·
`verify-dr-manifests` · `verify-dr-cache-512` · `verify-dr-stage-c` (C2's segmenter).
**Optional, for the raw-file check in section 6:** the Messidor-2 image mirror
(`mariaherrerot/messidor2preprocess`).

| Setting | Value |
|---|---|
| Accelerator | **None** — CPU |
| Persistence | Files only |
| Internet | On |

About 10–20 minutes.

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
sys.path.insert(0, str(REPO_DIR / "src"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Drop verify_dr modules left over from an earlier checkout in this kernel.
# Python caches modules by NAME, not by file, so re-cloning mid-session does
# nothing for an already-imported package: a later cell importing a function
# added upstream still fails with ImportError against the new files on disk.
for _stale in [m for m in list(sys.modules)
               if m == "verify_dr" or m.startswith("verify_dr.")]:
    del sys.modules[_stale]

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · Helpers, and every input found by name

In [ ]:
from pathlib import Path
from collections import Counter
import json, shlex, subprocess, sys, time, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a script, streaming its output live.

    The earlier notebooks use capture_output=True, which is fine for a two-minute
    manifest build and useless here: you would see nothing at all until a
    three-hour job exited. Streaming means a per-epoch line appears as it happens,
    so a run that is going wrong can be stopped in epoch 1 rather than hour 3.
    """
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        if code == 2:
            print("\nExit 2 is an argument error. Usually the cloned scripts are stale:")
            print("re-run the clone cell at the top, then run from there.")
        raise RuntimeError(f"exit {code}")

def cache_roots():
    """Every directory that looks like a build_cache.py output root.

    build_cache.py writes <root>/<dataset>/cache_report.json, so a report file
    identifies its root two levels up. /kaggle/input is searched first: a stale
    copy in /kaggle/working must never silently win over the dataset you attached.
    """
    found = []
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for root, _ in Counter(r.parent.parent for r in base.rglob("cache_report.json")).most_common():
            found.append(root)
    return found

def resolve_datasets(roots):
    """dataset -> the root holding the best copy of it.

    The cache is legitimately split across published datasets: the full build
    plus a later top-up. IDRiD's lesion masks ship as their own dataset
    ('verify-dr-idrid-masks'), so a single root shows idrid with zero mask
    channels even when the masks are attached. When a dataset appears in more
    than one root, the copy with more mask channels wins.
    """
    best = {}
    for root in roots:
        for d in sorted(x for x in root.iterdir() if x.is_dir()):
            if not (d / "cache_report.json").exists():
                continue
            masks = d / "masks"
            score = len(list(masks.iterdir())) if masks.is_dir() else 0
            if d.name not in best or score > best[d.name][1]:
                best[d.name] = (root, score)
    return {name: root for name, (root, _) in best.items()}

def extract_cache(dest=WORK / "cache512"):
    """Extract a cache published as a zip. Idempotent within a session.

    This costs GPU-session minutes, which come out of the 30 h/week quota. If you
    hit it every run, re-upload the cache to Kaggle as a *dataset* rather than as
    notebook output -- an uploaded zip is unpacked by Kaggle once, server-side.
    """
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
        except (zipfile.BadZipFile, OSError):
            continue
        if not any(n.endswith("cache_report.json") for n in names):
            continue
        marker = dest / ".extracted_from"
        if marker.exists() and marker.read_text().strip() == z.name:
            print(f"already extracted from {z.name}")
            return dest
        print(f"extracting {z.name} ({z.stat().st_size / 2**30:.1f} GB) -> {dest}")
        dest.mkdir(parents=True, exist_ok=True)
        started = time.time()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        marker.write_text(z.name)
        print(f"extracted in {(time.time() - started) / 60:.1f} min")
        return dest
    return None

def find_manifest_dir():
    """Phase 2's output: the directory holding dataset_plan.json and the variants."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

In [ ]:
from verify_dr.triage.params import load_params

PARAMS_PATH = REPO_DIR / "preregistration" / "fitted_params.json"
PARAMS = load_params(PARAMS_PATH)
OUT = WORK / "phase8"
OUT.mkdir(parents=True, exist_ok=True)
JOBS = [("eyepacs_test", "eyepacs_full.csv"), ("aptos", "aptos_external.csv"),
        ("messidor2", "messidor2_external.csv")]

def find_results():
    """10's output: results/<name>/results.json for each locked set."""
    found = {}
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for hit in sorted(base.rglob("results.json")):
            name = hit.parent.name
            if name in dict(JOBS) and name not in found:
                found[name] = hit
    return found

def find_locked():
    """09's output: a directory holding messidor2/, aptos/ and eyepacs_test/."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for hit in sorted(base.rglob("eyepacs_test/run.json")):
            root = hit.parent.parent
            if all((root / job / "run.json").exists() for job in ("aptos", "messidor2")):
                return root
    return None

def find_fitted():
    """08's output: the directory whose ood/ holds every model's statistics."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for hit in sorted(base.rglob("ood")):
            if all((hit / PARAMS["models"][m]["ood"]["file"].split("/", 1)[1]).exists()
                   for m in PARAMS["models"]):
                return hit.parent
    return None

def find_segmenter():
    hits = sorted(INPUT.rglob("C2_lesions/best.pt")) if INPUT.exists() else []
    return hits[0] if hits else None

def find_messidor2_raw():
    """The raw mirror's image folder (dataset card: messidor-2/messidor-2/preprocess)."""
    if not INPUT.exists():
        return []
    return sorted(d for d in INPUT.rglob("preprocess")
                  if d.is_dir() and "messidor" in str(d).lower())[:1]

ROOTS = cache_roots()
if not ROOTS and extract_cache():
    ROOTS = cache_roots()
MANIFEST_DIR = find_manifest_dir()
RESULTS_JSON = find_results()
LOCKED, FITTED, M2, RAW = find_locked(), find_fitted(), find_segmenter(), find_messidor2_raw()

missing = [what for what, ok in (
    ("verify-dr-results (10's output)", len(RESULTS_JSON) == len(JOBS)),
    ("verify-dr-locked (09's output)", LOCKED is not None),
    ("verify-dr-fitted (08's output)", FITTED is not None),
    ("verify-dr-manifests", MANIFEST_DIR is not None),
    ("verify-dr-cache-512", bool(ROOTS)),
    ("verify-dr-stage-c (C2's segmenter)", M2 is not None)) if not ok]
if missing:
    raise RuntimeError("Attach under Add Data: " + ", ".join(missing))
for name, path in RESULTS_JSON.items():
    res = json.loads(path.read_text())
    print(f"  {name:<13} results.json: unblinded={res['unblinded']}  rehearsal={res['rehearsal']}"
          f"  params {res['params_digest'][:12]}  ({res['created_at']})")
print("locked pass :", LOCKED)
print("fitted stats:", FITTED)
print("manifests   :", MANIFEST_DIR)
print("cache roots :", [str(r) for r in ROOTS])
print("segmenter   :", M2)
print("raw Messidor-2:", RAW[0] if RAW else "not attached -- section 6 skips the raw-file check")

## 3 · The secondary outcomes (P8.1): transcribed, nothing new computed

Read straight from `results.json`: the numbers the unblinding wrote but its readout did
not print. Mean [min–max] over each variant's three seeds.

In [ ]:
run(" ".join([f"python {q(REPO_DIR / 'scripts/phase8.py')} secondary --results"]
             + [q(RESULTS_JSON[name]) for name, _ in JOBS] + [f"--out {q(OUT)}"]))

## 4 · G1 and G3 on each set (P8.0, P8.2–P8.5): exploratory

First the reproduction gate: every per-image signal is recomputed as `analyse.py`
computed it, and nothing prints unless all five arms' AUCs match `results.json` for every
model and both analyses. Then:
- **G1**: where M1's errors go under each arm;
- **G1**: whether disagreement separates errors at fixed confidence;
- **G3**: the errors disagreement cannot see, and the correct calls M3 contradicts.

In [ ]:
for name, manifest in JOBS:
    print("\n" + "=" * 72 + f"\n{name}\n" + "=" * 72, flush=True)
    run(" ".join([f"python {q(REPO_DIR / 'scripts/phase8.py')} errors",
                  f"--pass-dir {q(LOCKED / name)}", f"--labels {q(MANIFEST_DIR / manifest)}",
                  "--split test", f"--fitted {q(PARAMS_PATH)}", f"--ood-dir {q(FITTED)}",
                  f"--results {q(RESULTS_JSON[name])}", f"--out {q(OUT / name)}"]))

## 5 · Figure 1 candidates (P8.5)

Model `eyepacs_full_s42`, amended evidence, most confident first:
- **A**: M1 says 0 confidently, M3 finds microaneurysms, and the truth is ≥ 1. This is figure 1.
- **B**: M1 says 0 confidently and is right, but M3 says 2.
- **C**: M1 says 2 and is right, but M3 finds nothing.

Each pair shows the image, then the same image with M2's lesion outlines.

In [ ]:
from IPython.display import Image as ShowImage

for name, _ in JOBS:
    cands = OUT / name / "candidates.csv"
    if not cands.exists():
        continue
    run(" ".join([f"python {q(REPO_DIR / 'scripts/phase8.py')} figures",
                  f"--candidates {q(cands)}", "--cache-root", *[q(r) for r in ROOTS],
                  f"--evidence-checkpoint {q(M2)}", f"--out {q(OUT / 'figures' / name)}"]))
    for sheet in sorted((OUT / "figures" / name).glob("*.png")):
        if "__" not in sheet.name:
            print(f"\n{name} / {sheet.stem}")
            display(ShowImage(filename=str(sheet), width=900))

## 6 · Messidor-2's preprocessing (P8.6): label-free

The dataset card's open question: has the mirror already been contrast-normalised or
Ben-Graham processed? A Ben-Graham image is mostly mid-grey, with little saturation and a
lot of high-pass energy, and its padding is grey rather than black.

In [ ]:
from IPython.display import Image as ShowImage

run(" ".join([f"python {q(REPO_DIR / 'scripts/phase8.py')} audit",
              f"--manifest-dir {q(MANIFEST_DIR)}", "--cache-root", *[q(r) for r in ROOTS],
              f"--locked {q(LOCKED)}", *(["--raw-root", *[q(r) for r in RAW]] if RAW else []),
              f"--out {q(OUT / 'audit')}"]))
for png in ("contact_sheet.png", "raw_messidor2.png"):
    path = OUT / "audit" / png
    if path.exists():
        print("\n" + png)
        display(ShowImage(filename=str(path), width=1100))

---
## 7 · Save, and what to paste back

1. **Save Version → Quick Save.**
2. Publish `/kaggle/working/phase8` as **`verify-dr-phase8`**. It holds `secondary.json`,
   `phase8.json` and `candidates.csv` per set, the figure sheets and `audit.json`.
3. Paste back the outputs of **sections 3, 4 and 6**. From section 5, download the
   sheets under `phase8/figures/eyepacs_test/` and send the one that should become
   figure 1 (category A), plus any that look wrong.